# 10-3절 연습 문제 풀이

이 노트북은 10-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch10/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 10-2/10-3절 공통 - 인코더/디코더 전용 트랜스포머
import math, random as _r
from torch.utils.data import Dataset, DataLoader
PAD, SOS, EOS, UNK, CLS = '<pad>', '<sos>', '<eos>', '<unk>', '[CLS]'

class DateValidator(nn.Module):
    """인코더만 사용하는 트랜스포머 (BERT 계열)"""
    def __init__(self, vocab, d_model=128, nhead=4, layers=2, pooling='cls'):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model, padding_idx=0)
        self.pos = nn.Embedding(64, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4,
                                               batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, layers)
        self.fc = nn.Linear(d_model, 2)
        self.pooling = pooling
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.encoder(self.emb(x) + self.pos(pos),
                         src_key_padding_mask=(x == 0))
        if self.pooling == 'cls':
            pooled = h[:, 0]                       # [CLS] 토큰 위치
        else:
            mask = (x != 0).unsqueeze(-1).float()  # <pad> 제외 평균
            pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(pooled)

class OzWriterTransformer(nn.Module):
    """디코더만 사용하는 트랜스포머 (GPT 계열)"""
    def __init__(self, vocab, d_model=256, nhead=4, layers=4, max_len=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4,
                                           batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers)
        self.fc = nn.Linear(d_model, vocab)
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1),
                                                             device=x.device)
        h = self.blocks(self.emb(x) + self.pos(pos), mask=mask, is_causal=True)
        return self.fc(h)

## 연습 10-12

온도 샘플링에 Top-k 샘플링을 결합한 generate_topk() 함수를 구현해 보자. Top-k 샘플링은 온도 샘플링에서 로짓 기준 상위 k개의 토큰만 남기고 나머지는 제외해 확률을 계산한 뒤 샘플링하는 방법이다. 예를 들어 로짓이 5, 3, 2인 세 토큰이 있고 k=2일 때 Top-2 샘플링은 2를 제외한 5와 3으로만 확률을 계산한다. k 값을 5, 10, 20으로 바꿔 가며 결과를 확인해 본 후, k 값에 따른 결과를 설명해 보자.

In [ ]:
@torch.no_grad()
def generate_topk(model, start_ids, vocab_size, k=10, temperature=1.0,
                  max_len=50, eos_id=None):
    model.eval()
    ids = start_ids.clone()
    for _ in range(max_len):
        logits = model(ids)[:, -1] / temperature
        topk = logits.topk(k, dim=-1)                    # 상위 k개만 남긴다
        filtered = torch.full_like(logits, -float('inf'))
        filtered.scatter_(1, topk.indices, topk.values)
        probs = torch.softmax(filtered, dim=-1)
        nxt = torch.multinomial(probs, 1)
        ids = torch.cat([ids, nxt], dim=1)
        if eos_id is not None and nxt.item() == eos_id: break
    return ids

m = OzWriterTransformer(100).to(device)
start = torch.randint(0, 100, (1, 5), device=device)
print(f'Top-k 생성 결과 형태: {tuple(generate_topk(m, start, 100, k=5, max_len=10).shape)}')

Top-k 샘플링은 확률이 낮은 꼬리 토큰을 **후보에서 아예 제외**한 뒤 정규화한다. 온도만 쓰면 확률이 아주 낮은 엉뚱한 토큰도 가끔 뽑히는데, Top-k가 그 위험을 막는다. `-inf`로 채운 뒤 소프트맥스를 적용하면 제외한 토큰의 확률이 정확히 0이 된다.

## 연습 10-13

빔 너비가 3 이상일 때 반복 루프가 심해지는 현상을 누적 로그 확률의 관점에서 분석해 보자. 그런 다음 같은 토큰이 연속해 등장하면 해당 토큰의 로짓에 페널티를 부과하는 반복 페널티 보정을 [코드 10-15]의 generate_beam_search() 함수에 적용해, 페널티 강도에 따른 생성 결과 변화를 비교해 보자.

### 풀이

**빔 너비가 클수록 반복이 심해지는 이유**

빔 서치는 누적 로그 확률이 가장 높은 후보를 남긴다. 그런데 이미 등장한 토큰을 다시 내는 것은 모델 입장에서 **확률이 높은 안전한 선택**이다. 새로운 토큰은 확률이 분산되어 로그 확률이 낮게 잡힌다.

빔이 넓어지면 이런 '안전한 반복' 경로가 여러 개 살아남아 상위 후보를 모두 차지한다. 결국 다양한 후보를 보려던 목적과 반대로 **비슷한 반복 문장만 남는다**.

In [ ]:
def apply_repetition_penalty(logits, generated_ids, penalty=1.2):
    """이미 생성된 토큰의 로짓에 페널티를 적용한다."""
    for token_id in set(generated_ids):
        if logits[token_id] > 0:
            logits[token_id] /= penalty      # 양수는 나눠서 줄이고
        else:
            logits[token_id] *= penalty      # 음수는 곱해서 더 낮춘다
    return logits

logits = torch.tensor([2.0, -1.0, 0.5, 3.0])
print(f'원래   : {logits.tolist()}')
print(f'페널티 : {apply_repetition_penalty(logits.clone(), [0, 3]).tolist()}')

로짓이 양수인지 음수인지에 따라 처리를 달리하는 것이 요령이다. 양수를 그냥 곱하면 오히려 커지므로 나누고, 음수는 곱해서 더 작게 만든다. 이 방식은 허깅페이스 `generate()`의 `repetition_penalty`와 같은 구현이다.

## 연습 10-14

빔 서치와 온도 샘플링을 결합해 생성하는 generate() 함수를 구현해 보자. 다양한 빔 너비와 온도값의 조합을 실험해 보면서 가장 좋다고 생각하는 각자의 조합값을 찾아 보자. 만약 여전히 결과가 마음에 들지 않는다면, 그 한계가 어디서 왔는지도 생각해 보자.

In [ ]:
@torch.no_grad()
def generate_beam_sampling(model, start_ids, beam_width=3, temperature=0.8,
                           max_len=30, eos_id=None):
    """빔 서치의 후보 확장에 온도 샘플링을 결합"""
    model.eval()
    beams = [(start_ids, 0.0)]
    for _ in range(max_len):
        candidates = []
        for ids, score in beams:
            logits = model(ids)[:, -1] / temperature
            probs = torch.softmax(logits, dim=-1)
            picks = torch.multinomial(probs, beam_width)      # 확률적으로 뽑는다
            for i in range(beam_width):
                nxt = picks[:, i:i + 1]
                lp = torch.log(probs[0, nxt.item()] + 1e-12).item()
                candidates.append((torch.cat([ids, nxt], 1), score + lp))
        # 길이 정규화 점수로 상위 beam_width개 유지
        beams = sorted(candidates, key=lambda b: b[1] / b[0].size(1),
                       reverse=True)[:beam_width]
    return max(beams, key=lambda b: b[1] / b[0].size(1))[0]

out = generate_beam_sampling(m, start, beam_width=3, max_len=10)
print(f'빔+샘플링 결과 형태: {tuple(out.shape)}')

빔 서치의 **탐색력**과 샘플링의 **다양성**을 결합한 방식이다. 후보 확장을 확률적으로 하되 유지 단계는 점수 기준으로 하므로, 순수 빔 서치보다 반복이 줄고 순수 샘플링보다 문장이 안정적이다.

빔 너비 3~5, 온도 0.7~0.9 조합이 대체로 무난하다.

## 연습 10-15

OzWriterTransformer를 [코드 10-12]의 비중첩 분할 대신, 7장 OzWriter의 슬라이딩 윈도우 데이터셋(한 토큰씩 이동)으로 학습해 보자. 같은 에포크 수에서 두 방식의 학습 곡선과 생성 품질을 비교하고, 본문에서 설명한 트레이드오프(슬라이딩 윈도우의 중복 학습 대 비중첩 분할의 경계 토큰 짧은 문맥)가 실제로 어떻게 나타나는지 분석해 보자.

### 풀이

**비중첩 분할**은 텍스트를 시퀀스 길이만큼 잘라 겹치지 않게 사용한다. 샘플 수가 적어 한 에포크가 빠르지만, 경계에 걸친 문맥은 학습되지 않는다.

**슬라이딩 윈도우**(한 토큰씩 이동)는 같은 텍스트에서 훨씬 많은 샘플을 만든다. 모든 위치가 시작점이 되므로 문맥 학습이 촘촘해지고 생성 품질이 좋아진다. 대신 한 에포크가 시퀀스 길이 배만큼 길어지고, 겹치는 내용이 많아 과적합도 빨라진다.

**같은 에포크 수로 비교하면** 슬라이딩 윈도우 쪽이 손실이 훨씬 빨리 내려간다. 실제로는 본 데이터의 양이 다르므로, 공정한 비교는 **에포크가 아니라 처리한 토큰 수**를 기준으로 해야 한다.

## 연습 10-16

[도전 문제] OzWriterTransformer 모델의 입력 토큰을 단어 단위에서 글자 단위로 바꿔 같은 데이터셋을 학습해 보자. 어휘 사전 크기와 시퀀스 길이가 어떻게 달라져야 하는지, 학습 안정성과 생성 품질이 어떻게 달라지는지 분석한 후, 토큰화 단위 선택이 트랜스포머 기반 생성 모델에 미치는 영향을 자신의 말로 정리해 보자.

### 풀이

**어휘 사전 크기**: 단어 단위는 수천~수만 개지만, 글자 단위는 영어 기준 약 60~100개로 크게 줄어든다. 임베딩과 출력층 파라미터가 대폭 감소한다.

**시퀀스 길이**: 같은 문장을 표현하는 데 토큰이 4~5배 필요하다. 트랜스포머의 어텐션은 길이의 **제곱**에 비례하므로 연산량이 크게 늘고, `max_len`과 위치 인코딩 범위도 함께 키워야 한다.

**학습 안정성과 생성 품질**: 글자 단위는 처음 보는 단어도 만들 수 있고 오타에 강하지만, 철자를 하나하나 배워야 해 학습이 오래 걸리고 초반에는 단어조차 제대로 만들지 못한다. 문법 수준의 일관성은 단어 단위가 낫다.

정리하면 **어휘는 줄고 길이는 늘어나는 트레이드오프**다. 그래서 실무에서는 둘의 중간인 서브워드 토큰화를 쓴다.

## 연습 10-17

[도전 문제] 문자열 형태의 정수를 오름차순으로 정렬하는 모델([연습 문제 10-1] 참고)을 디코더만 사용하는 트랜스포머로 만들 수 있을지, 만들 수 있다면 어떤 방법이 필요할지 직접 답해 보고 모델로 구현해 보자.

10장 학습 노트

### 풀이

**만들 수 있다.** 디코더 전용 트랜스포머는 '다음 토큰 예측'만 하므로, 입력과 출력을 **하나의 시퀀스로 이어 붙이면** 된다.

```
5, 724, 223  <sep>  5, 223, 724 <eos>
└─ 입력부 ─┘        └─ 출력부 ─┘
```

필요한 처리는 세 가지다.

1. **구분 토큰(`<sep>`)** 을 두어 입력이 끝나고 출력이 시작되는 지점을 알린다.
2. **손실 마스킹**: 입력부는 예측 대상이 아니므로 정답 레이블을 `-100`으로 두어 손실에서 제외한다(12장·13장에서 쓰는 기법과 동일).
3. **추론**: 입력부와 `<sep>`까지 넣고 그다음부터 자기회귀로 생성한다.

이것이 바로 GPT 계열 모델이 번역·요약 같은 Seq2Seq 과제를 인코더 없이 수행하는 방식이며, 12장 LLM의 프롬프트 방식과 같은 원리다.